# Iniciando o Spark

In [ ]:
## Bloco de codigo para instalar versao especifica dos pacotes
!pip install pyspark

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Trusted_base_bureau") \
    .getOrCreate()

# Importando bibliotecas

In [ ]:
import os
import pytz
import datetime
from datetime import datetime
#from pyspark.sql.types import *
#from pyspark.sql.functions import count, avg
#import sys
#import numpy as np
#from datetime import datetime
#from pyspark.sql import SQLContext
#from datetime import timedelta
#from datetime import date
#from dateutil.relativedelta import relativedelta
#from pyspark.sql.functions import udf, lpad, translate

# Funções auxiliares e variáveis

In [ ]:
# Função de log
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + " >>>"

# Timestamp de processamento (com hora/minuto/segundo)
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Data da execução (AAAAmmdd)
PROCESS_DATE = datetime.now().strftime("%Y%m%d")

# Período de referência (AAAAmm)
REF_PERIOD = datetime.now().strftime("%Y%m")

#Alterar o path_padrao caso seus arquivos não estejam nesse mesmo caminho
path_padrao = "/content/gdrive/Othercomputers/Meu laptop"

# Buckets e nomes de saída
bucket_base = "base_bureau"
bucket_raw = f"{path_padrao}/Database_raw/base_score_bureau_movel_full"
bucket_trusted = f"{path_padrao}/Database_trusted/base_score_bureau_movel_full"
bucket_control = f"{path_padrao}/Database_control/base_score_bureau_movel_full"
output_trusted = f"trusted_{bucket_base}"

# Prints para conferência
print("PROCESS_DATE:", PROCESS_DATE)
print("REF_PERIOD:", REF_PERIOD)
print("dthproc:", dthproc)
print("bucket_raw:", bucket_raw)
print("bucket_trusted:", bucket_trusted)
print("bucket_control:", bucket_control)

PROCESS_DATE: 20260210
REF_PERIOD: 202602
dthproc: 20260210083715
bucket_raw: /content/gdrive/Othercomputers/Meu laptop/Database_raw/base_score_bureau_movel_full
bucket_trusted: /content/gdrive/Othercomputers/Meu laptop/Database_trusted/base_score_bureau_movel_full
bucket_control: /content/gdrive/Othercomputers/Meu laptop/Database_control/base_score_bureau_movel_full


# Leitura dos dados na camada Raw

In [ ]:
path_raw = bucket_raw
df_raw = spark.read.parquet(path_raw)
df_raw.createOrReplaceTempView("raw_base_bureau")

print(log(), "Registros na Raw:", df_raw.count())
df_raw.show(5, truncate=False)


2026-02-10 11:59:45 >>> Registros na Raw: 3795310
+------+---------------+----+----+---------+--------+--------+-----------+
|SAFRA |FLAG_INSTALACAO|FPD |PROD|flag_mig2|SCORE_01|SCORE_02|NUM_CPF    |
+------+---------------+----+----+---------+--------+--------+-----------+
|202410|1              |1   |CMV |Aquisição|2       |1       |ZZZZZZZX7T9|
|202410|0              |NULL|CMV |NULL     |562     |559     |ZZZZZZZ8TZ8|
|202410|0              |NULL|CMV |NULL     |585     |559     |ZZZZZZW9XWN|
|202410|1              |0   |CMV |PRE      |562     |636     |ZZZZZX7XWY8|
|202410|1              |1   |CMV |Aquisição|538     |570     |ZZZZZX8TTUZ|
+------+---------------+----+----+---------+--------+--------+-----------+
only showing top 5 rows


# Processamento tipagem para camada Trusted

In [ ]:
df_trusted = spark.sql(f"""
    SELECT
        '{dthproc}' AS ts_proc,
        '{dthproc}' AS ts_proc_partition,
        -- SAFRA convertida para DATE (primeiro dia do mês)
        CAST(CONCAT(SUBSTRING(SAFRA, 1, 4), '-', SUBSTRING(SAFRA, 5, 2), '-01') AS DATE) AS SAFRA,

        -- Ano e Mês extraídos da SAFRA
        CAST(SUBSTRING(SAFRA, 1, 4) AS INT) AS Ano,
        CAST(SUBSTRING(SAFRA, 5, 2) AS INT) AS Mes,

        CAST(FLAG_INSTALACAO AS BOOLEAN) AS FLAG_INSTALACAO,
        CAST(PROD AS STRING) AS ProductDescription,
        CAST(flag_mig2 AS STRING) AS ProductMigration,
        CAST(SCORE_01 AS FLOAT) AS Score01,
        CAST(SCORE_02 AS FLOAT) AS Score02,
        CAST(FPD AS INT) AS FPD,
        CAST(NUM_CPF AS STRING) AS NUM_CPF
    FROM raw_base_bureau
""")

df_trusted.createOrReplaceTempView("lake_base_bureau")
df_trusted.cache()

print(log(), "Registros Trusted:", df_trusted.count())
#df_trusted.printSchema()
df_trusted.show(5, truncate=False)

2026-02-10 11:44:36 >>> Registros Trusted: 3795310
+--------------+-----------------+----------+----+---+--------------+------------------+----------------+-------+-------+----+-----------+
|ts_proc       |ts_proc_partition|SAFRA     |Ano |Mes|IsInstallation|ProductDescription|ProductMigration|Score01|Score02|FDP |NUM_CPF    |
+--------------+-----------------+----------+----+---+--------------+------------------+----------------+-------+-------+----+-----------+
|20260210083715|20260210083715   |2024-10-01|2024|10 |true          |CMV               |Aquisição       |2.0    |1.0    |1   |ZZZZZZZX7T9|
|20260210083715|20260210083715   |2024-10-01|2024|10 |false         |CMV               |NULL            |562.0  |559.0  |NULL|ZZZZZZZ8TZ8|
|20260210083715|20260210083715   |2024-10-01|2024|10 |false         |CMV               |NULL            |585.0  |559.0  |NULL|ZZZZZZW9XWN|
|20260210083715|20260210083715   |2024-10-01|2024|10 |true          |CMV               |PRE             |562.0  |63

# Salvar na camada Trusted

In [ ]:
path_trusted = os.path.join(bucket_trusted, output_trusted)
print("Trusted path:", path_trusted)

df_trusted.write \
    .partitionBy("SAFRA","ts_proc_partition") \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(path_trusted)

Trusted path: /content/gdrive/Othercomputers/Meu laptop/Database_trusted/base_score_bureau_movel_full/trusted_base_bureau


# Controle de carga

In [ ]:
controle = spark.sql(f"""
    SELECT
        '{output_trusted}' AS name_file,
        ts_proc,
        ts_proc_partition,
        COUNT(*) AS qtd_registros
    FROM lake_base_bureau
    GROUP BY 1,2,3
""")

controle.createOrReplaceTempView("controle")
controle.cache()

print(log(), "Registros controle:", controle.count())
controle.show(truncate=False)

2026-02-10 11:50:59 >>> Registros controle: 1
+-------------------+--------------+-----------------+-------------+
|name_file          |ts_proc       |ts_proc_partition|qtd_registros|
+-------------------+--------------+-----------------+-------------+
|trusted_base_bureau|20260210083715|20260210083715   |3795310      |
+-------------------+--------------+-----------------+-------------+



# Controle de processamento

In [ ]:
path_control = os.path.join(bucket_control, f"tb_0002_controle_processamento_{bucket_base}_trusted")
print("Control path:", path_control)

controle.write \
    .mode("append") \
    .option("compression", "snappy") \
    .parquet(path_control)

Control path: /content/gdrive/Othercomputers/Meu laptop/Database_control/base_score_bureau_movel_full/tb_0002_controle_processamento_base_bureau_trusted
